# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">4주차 · 사람이 정한 규칙을 지우고 데이터가 찾은 취향으로 추천하기</mark>

지난 세 주 동안 추천의 규칙은 **사람이 정했습니다.** 주력 섹터로 나누자, 평균 위험도로 나누자 — 강사가 고른 것입니다.

오늘은 **아무도 정하지 않습니다.** 거래 기록만 주고 **모델이 스스로 찾게** 합니다. 그 방법이 **행렬분해**입니다.

---

### 오늘의 구성

| 파트 | 종류 | 어디서 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 슬라이드 | 상호작용 행렬 · 저차원 근사 · SVD · 내적을 이해한다 |
| 2 | 실습 | **이 노트북 4.2~4.6** | 행렬을 만들고, 모델을 학습하고, 추천 결과를 채점한다 |
| 3 | 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

> **슬라이드에서 원리를 이해하고, 노트북에서는 실제로 작동하는지만 확인합니다.** 채점은 3주차와 같은 `recsys.recall_per_user`를 그대로 사용합니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/04_matrix_factorization.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 실습 — 모델에게 취향을 찾게 한다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 오늘 나오는 말  `[PPT]`</mark>

1. **상호작용 행렬 (interaction matrix)** — 세로가 사람, 가로가 종목인 표. 담은 칸에 **1**, 나머지 칸에 **0**을 적습니다
2. **행렬분해 (matrix factorization)** — 큰 행렬을 작은 요인 행렬로 나눠 추천 점수를 만듭니다
3. **잠재 요인 (latent factor)** — 사람과 종목의 행동 패턴을 설명하는 축. 오늘은 **8개**를 사용합니다
4. **Recall@10** — 추천 10개가 실제로 담은 종목을 얼마나 맞혔는지 재는 점수입니다


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 데이터와 채점 (지난주 그대로)</mark>

**여기서 하는 일** — 데이터를 읽고, 채점에 쓸 것 두 개만 만들어 둡니다.

**채점 함수는 다시 만들지 않습니다.** `recsys.recall_per_user` 가 이미 있습니다 — 3주차에 쓰던 그 방식 그대로(시간순 분할 · 이미 담은 것은 정답에서 빼기 · 맞힐 것이 없으면 채점에서 빼기)입니다.

**잣대가 같아야** 오늘 숫자를 2·3주차 0.2684 와 견줄 수 있습니다.

**쉽게 말하면 다음과 같습니다.**
1. 종목·투자자·선택 기록 세 표를 읽습니다.
2. 기록을 과거의 **학습 구간**과 마지막 한 달의 **채점 구간**으로 나눕니다.
3. 이미 담은 종목과 전체 인기순위를 미리 정리합니다.
4. 마지막에 지난주와 똑같은 채점 함수로 점수를 잽니다.

In [2]:
# 이 셀에서 하는 일 — 오늘 쓸 도구를 불러온다
# 왜 하나 — pandas 로 표를, numpy 로 숫자 묶음을 다루고, recsys.py 에는 1~3주차 함수가 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import recsys                               # 이 수업용으로 만든 도구 모음

print("도구 준비 완료 · pandas", pd.__version__, "· numpy", np.__version__)   # 버전도 남겨 둔다

도구 준비 완료 · pandas 3.0.5 · numpy 2.5.2


In [3]:
# 이 셀에서 하는 일 — 데이터를 읽고 3주차와 똑같이 학습 구간·채점 구간으로 나눈다
# 왜 하나 — 잣대가 지난주와 같아야 0.2684 와 견줄 수 있다

items, users, interactions = recsys.load()          # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 1주차부터 쓰던 시간순 분할
print(f"학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

이름_사전 = items.set_index("item_id")["name"].to_dict()       # {종목 번호: 이름}
위험도_사전 = items.set_index("item_id")["risk_level"].to_dict()  # {종목 번호: 1~5}

이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # {사람: 학습 구간에 담은 집합}
전체_인기순위 = list(train["item_id"].value_counts().index)              # 많이 담긴 순 (기록 없는 사람용)
print(f"학습 구간에 기록이 있는 사람 {len(이미_담은것)}명 · 인기순위 {len(전체_인기순위)}개")
print("채점은 recsys.recall_per_user 로 합니다 — 3주차와 재는 방식이 같습니다.")

학습 구간 7,095건 · 채점 구간 991건 · 자른 날짜 2026-07-30
학습 구간에 기록이 있는 사람 285명 · 인기순위 100개
채점은 recsys.recall_per_user 로 합니다 — 3주차와 재는 방식이 같습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 사람 × 종목 표를 만든다</mark>

**4.2 에서 하는 일** — 거래 기록을 **표 한 장**으로 바꿉니다.

**1. 왜 표로 바꾸나**
- 거래 기록은 「누가 · 무엇을 · 언제」가 한 줄씩 적힌 **긴 목록**입니다. 모델은 이 모양을 못 읽습니다.
- 세로 **사람**, 가로 **종목**인 표로 바꿔 줘야 합니다. 담았으면 **1**, 안 담았으면 **0**.

**2. 이 표의 빈칸이 오늘의 과녁입니다**
- 칸이 285 × 100 = **28,500개**인데 1 이 들어가는 칸은 **7,095개**뿐입니다.
- 이 **0 은 「싫다」가 아니라 「아직 모른다」**입니다. 그 빈칸을 그럴듯한 숫자로 채우는 것이 오늘 할 일입니다.

**쉽게 말하면 다음과 같습니다.**
1. 사람 한 명을 한 줄로 놓습니다.
2. 종목 하나를 한 칸으로 놓습니다.
3. 그 사람이 담은 종목에는 1, 나머지에는 0을 적습니다.
4. 모델이 읽을 수 있는 **사람 285명 × 종목 100개 표**가 완성됩니다.

In [5]:
# 이 셀에서 하는 일 — 진짜 데이터로 사람 × 종목 표를 만든다
# 왜 하나 — 이 표가 오늘 쪼갤 대상이다. 학습 구간만 쓴다 — 채점 구간을 보면 반칙이니까

표 = pd.crosstab(train["user_id"], train["item_id"])      # 학습 구간만으로 펼친다
표 = (표 > 0).astype(int)                                  # 담았으면 1, 아니면 0
표 = 표.reindex(columns=items["item_id"], fill_value=0)    # 아무도 안 담은 종목도 열로 세운다

칸_전체 = 표.shape[0] * 표.shape[1]                        # shape = (세로, 가로)
칸_채움 = int(표.values.sum())                             # 1 이 들어간 칸의 개수
print(f"표 크기 — 사람 {표.shape[0]}명 × 종목 {표.shape[1]}개 = 칸 {칸_전체:,}개")
print(f"1 인 칸 {칸_채움:,}개 = 전체의 {칸_채움 / 칸_전체:.1%}   ← 나머지는 전부 0")

print("\n왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)")   # 표가 실제로 어떻게 생겼나
print(표.iloc[:5, :6])

표 크기 — 사람 285명 × 종목 100개 = 칸 28,500개
1 인 칸 7,095개 = 전체의 24.9%   ← 나머지는 전부 0

왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)
item_id  I001  I002  I003  I004  I005  I006
user_id                                    
U0001       0     0     0     0     0     1
U0002       0     1     0     0     0     0
U0003       0     0     0     0     1     0
U0004       1     0     1     1     1     0
U0005       0     0     0     0     0     0


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 TruncatedSVD로 요인 8개를 남긴다</mark>

**4.3 에서 하는 일** — 28,500칸짜리 표를 **사람표(285×8)**와 **종목표(8×100)**로 줄입니다.

- 사용자 한 명마다 행동 패턴을 나타내는 숫자 **8개**가 생깁니다.
- 종목 하나에도 같은 축의 숫자 **8개**가 생깁니다.
- `TruncatedSVD`는 중요한 축 8개만 남겨, 외우는 대신 큰 패턴을 잡습니다.


In [6]:
# 이 셀에서 하는 일 — 28,500칸짜리 표를 사람표와 종목표로 쪼갠다
# 왜 하나 — 큰 표를 그대로 외우지 않고, 사람과 종목의 큰 행동 패턴만 남기려는 것이다

from sklearn.decomposition import TruncatedSVD     # 행렬분해 도구

모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개로 쪼갠다
사람표 = 모델.fit_transform(표)                         # 285 × 8 — 사람마다 숫자 8개
종목표 = 모델.components_                               # 8 × 100 — 종목마다 숫자 8개

print(f"쪼개기 전 — 칸 {표.shape[0] * 표.shape[1]:,}개")
print(f"쪼갠 뒤   — 사람표 {사람표.shape} + 종목표 {종목표.shape} = 칸 {사람표.size + 종목표.size:,}개")

쪼개기 전 — 칸 28,500개
쪼갠 뒤   — 사람표 (285, 8) + 종목표 (8, 100) = 칸 3,080개


**출력 읽는 법**

- 원래 표는 **28,500칸**이지만, 두 작은 표에는 **3,080칸**만 남았습니다.
- 사람 285명은 각자 숫자 8개로, 종목 100개도 각자 숫자 8개로 표현됩니다.
- 정보를 전부 외우지 않고 추천에 필요한 큰 패턴만 압축한 것입니다.

In [7]:
# 이 셀에서 하는 일 — 축의 중요도와 U0003의 성향 점수 8개를 확인한다
# 왜 하나 — 모델이 사람을 실제로 어떤 숫자로 표현했는지 눈으로 보려는 것이다

print("축마다 붙은 중요도 :", np.round(모델.singular_values_, 1))   # 클수록 더 중요한 축
print(f"8개 축이 원래 표를 설명하는 비율 : {모델.explained_variance_ratio_.sum():.1%}")

사람_이름들 = list(표.index)                            # 표의 세로 이름(사람) 목록
열이름 = list(표.columns)                               # 표의 가로 이름(종목 번호) 목록
자리 = 사람_이름들.index("U0003")                       # U0003 이 몇 번째 줄인지
print(f"\nU0003 의 잠재 요인 8개 : {np.round(사람표[자리], 2)}")

축마다 붙은 중요도 : [46.2 21.1 18.4 13.9 13.2 12.8 12.5 11.9]
8개 축이 원래 표를 설명하는 비율 : 33.9%

U0003 의 잠재 요인 8개 : [ 3.16 -1.38  1.43  0.66 -1.09  0.05  0.09  0.28]


**쉽게 말하면 다음과 같습니다.**

1. 사람들의 종목 선택 기록을 살펴봅니다.
2. 서로 비슷하게 움직이는 종목들을 찾아 **8가지 숨은 투자 성향**으로 묶습니다.
3. 각 사람을 8개의 성향 점수로 표현합니다.
4. 각 종목도 같은 8개의 성향 점수로 표현합니다.
5. 위의 `축마다 붙은 중요도`는 8가지 성향 중 무엇이 더 큰 패턴인지 보여줍니다.
6. 마지막 줄은 예시로 **U0003 사용자의 성향 점수 8개**를 보여줍니다.

숨은 성향에는 실제 이름이 붙어 있지 않지만, 다음과 같은 느낌일 수 있습니다.
- 대형주 선호
- 기술주 선호
- 안정적인 종목 선호
- 고위험 종목 선호

> 중요한 점: 모델은 이 이름을 배우는 것이 아니라 **함께 선택되는 패턴을 숫자로 찾아냅니다.** 이름은 사람이 결과를 보고 나중에 해석합니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.4 모든 추천 점수를 한 번에 만든다</mark>

**4.4 에서 하는 일** — 사람표와 종목표를 `@`로 곱해 **285명 × 100종목의 점수표**를 만듭니다.

- 사람과 종목의 같은 요인 값을 곱해 모두 더하면 추천 점수 하나가 됩니다.
- 코드에서는 이 계산을 칸마다 반복하지 않고 `사람표 @ 종목표` 한 줄로 처리합니다.
- 원래 0이었던 칸에도 점수가 생기므로, 아직 담지 않은 종목을 줄 세울 수 있습니다.

**쉽게 말하면 다음과 같습니다.**
1. 한 사람의 성향 점수 8개를 가져옵니다.
2. 한 종목의 성향 점수 8개를 가져옵니다.
3. 같은 자리끼리 곱하고 모두 더해 추천 점수 하나를 만듭니다.
4. 이 계산을 285명 × 100종목에 한 번에 적용합니다.
5. 점수가 높을수록 그 사람과 잘 맞을 가능성이 큰 종목입니다.

In [8]:
# 이 셀에서 하는 일 — 그 계산을 모든 칸에 한 번에 해서 점수표를 만든다
# 왜 하나 — 사람 285명 × 종목 100개를 하나씩 곱할 수는 없다. @ 하나면 끝난다

점수표 = 사람표 @ 종목표                                 # @ = 행렬 곱하기. 285 × 100 으로 돌아온다
print("점수표 크기 :", 점수표.shape, "  ← 원래 표와 같은 크기")

안_담은것 = []                                           # U0003 이 학습 구간에 안 담은 종목
for 번호 in 열이름:                                      # 종목을 하나씩
    if 표.loc["U0003", 번호] == 0:                       # 표에서 0 이면 안 담은 것
        안_담은것.append(번호)

점수_모음 = {}                                           # {종목 번호: 점수표에 붙은 점수}
for 번호 in 안_담은것:                                    # 안 담은 종목을 하나씩
    점수_모음[번호] = 점수표[자리][열이름.index(번호)]      # 그 칸의 점수를 꺼낸다

print(f"\nU0003 이 안 담은 종목 {len(안_담은것)}개 — 표에서는 전부 0 이었습니다")
print("점수표에는 숫자가 붙어 있습니다 (높은 순 세 개)")
for 번호 in sorted(점수_모음, key=점수_모음.get, reverse=True)[:3]:   # 큰 순으로 셋
    print(f"  {이름_사전[번호]:34} {점수_모음[번호]:.3f}   ← 추천 후보")

점수표 크기 : (285, 100)   ← 원래 표와 같은 크기

U0003 이 안 담은 종목 71개 — 표에서는 전부 0 이었습니다
점수표에는 숫자가 붙어 있습니다 (높은 순 세 개)
  TIGER 미국나스닥100레버리지                 0.641   ← 추천 후보
  Salesforce                         0.605   ← 추천 후보
  iShares Global Clean Energy ETF    0.603   ← 추천 후보


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.5 추천 10개를 고른다  ✏️ 직접 해 보기</mark>

**4.5 에서 하는 일** — 점수표에서 높은 순으로 10개를 뽑아 추천 목록을 만듭니다.

- 이미 담은 종목은 제외합니다.
- 학습 기록이 없는 신규 사용자는 전체 인기 목록을 받습니다.
- 직접 채울 부분은 점수가 높은 순서대로 자리 번호를 얻는 **한 줄**입니다.

**쉽게 말하면 다음과 같습니다.**
1. 한 사람의 종목 100개 점수를 큰 순서대로 정렬합니다.
2. 그 사람이 이미 담은 종목은 건너뜁니다.
3. 남은 종목 중 위에서 10개를 추천합니다.
4. 실행 뒤에는 U0003의 추천 10개와 위험도를 확인합니다.

In [10]:
# 이 셀에서 하는 일 — ✏️ 빈칸 1 · 점수가 높은 순으로 추천 10개를 고른다
# 왜 하나 — 여기가 오늘 만드는 추천의 심장이다. 나머지는 지난주 것을 그대로 쓴다

def 추천하기(사람, 이미):
    """그 사람의 점수표 한 줄을 보고 높은 순으로 10개를 돌려준다."""
    if 사람 not in 사람_이름들:                            # 표에 줄이 없는 사람(신규)은
        return recsys.take(전체_인기순위, 이미)[:10]        # 전체 인기 목록으로 준다

    내_점수 = 점수표[사람_이름들.index(사람)]               # 그 사람 줄 = 종목 100개의 점수
    내_순서 = []                                          # 점수가 높은 종목부터 담을 목록
    for 자리번호 in np.argsort(-내_점수):                  # ← ✏️ 빈칸 : 높은 순 자리 번호
        내_순서.append(열이름[자리번호])                    # 자리 번호를 종목 번호로 바꿔 담는다
    return recsys.take(내_순서, 이미)[:10]                 # 이미 담은 것을 빼고 위에서 10개


내_추천 = 추천하기("U0003", 이미_담은것["U0003"])           # 만들었으면 한 명 돌려 본다
print("U0003 에게 줄 추천 10개")
for 순위, 번호 in enumerate(내_추천, start=1):             # enumerate = 번호를 붙여 준다
    print(f"  {순위:2}위  {이름_사전[번호]:34} 위험도 {위험도_사전[번호]}")

U0003 에게 줄 추천 10개
   1위  TIGER 미국나스닥100레버리지                 위험도 5
   2위  Salesforce                         위험도 4
   3위  iShares Global Clean Energy ETF    위험도 4
   4위  ProShares UltraPro Short QQQ       위험도 5
   5위  Vanguard FTSE Developed Markets ETF 위험도 3
   6위  LG에너지솔루션                           위험도 5
   7위  Global X Lithium & Battery Tech ETF 위험도 5
   8위  Vanguard S&P 500 ETF               위험도 3
   9위  TIGER 200                          위험도 3
  10위  알테오젠                               위험도 5


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.6 채점한다 — 지난주 자를 그대로 쓴다</mark>

**4.6 에서 하는 일** — 방금 만든 추천 방식을 3주차와 같은 `Recall@10`으로 잽니다.

- 채점 코드를 다시 만들지 않고 **`recsys.recall_per_user` 한 줄**을 사용합니다.
- 재는 방법이 같으므로, 점수 차이는 추천 방식이 바뀐 결과입니다.

**쉽게 말하면 다음과 같습니다.**
1. 각 사용자에게 추천 10개를 만듭니다.
2. 마지막 한 달에 실제로 담은 종목과 비교합니다.
3. 사용자별로 맞힌 비율을 구한 뒤 평균냅니다.
4. **0.3918**이 나오면, 맞힐 수 있었던 종목 10개 중 평균 약 3.9개를 추천 목록에 넣었다는 뜻입니다.

In [11]:
# 이 셀에서 하는 일 — 3주차와 같은 자로 채점한다 (한 줄)
# 왜 하나 — 잣대를 다시 만들지 않는다. 같은 함수를 부르는 것이 「같은 자」라는 증거다

점수들 = recsys.recall_per_user(추천하기, train, test)     # {사람: Recall@10} — 3주차 방식 그대로
행렬분해_점수 = float(np.mean(list(점수들.values())))       # 점수판에 적을 값

print(f"4주차 행렬분해 — Recall@10 = {행렬분해_점수:.4f}   (채점 대상 {len(점수들)}명)")
print(f"2·3주차 세그먼트 — Recall@10 = 0.2684")             # 견줄 상대
print(f"\n→ {(행렬분해_점수 - 0.2684) / 0.2684:+.1%} 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.")

4주차 행렬분해 — Recall@10 = 0.3918   (채점 대상 266명)
2·3주차 세그먼트 — Recall@10 = 0.2684

→ +46.0% 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.


**결과 — 규칙을 안 정했더니 올랐습니다**

| 주차 | 규칙을 누가 정했나 | Recall@10 |
|---|---|---|
| 1주차 | 규칙이랄 것도 없음 (인기순) | 0.2163 |
| 2·3주차 | 사람 — 주력 섹터·평균 위험도 | 0.2684 |
| **4주차** | **아무도 안 정함 — 모델이 찾음** | **0.3918** |

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.7 오늘의 마무리  `[PPT]`</mark>

1. 사람 × 종목 상호작용 행렬을 만들었습니다.
2. `TruncatedSVD(k=8)`로 사용자와 종목의 요인 값을 만들었습니다.
3. 추천 10개를 뽑아 같은 기준으로 채점했고 **Recall@10 = 0.3918**을 확인했습니다.

다음 주에는 종목을 전부 재지 않고 후보를 먼저 추립니다. 점수가 아니라 **규모가 커져도 버티는 구조**를 얻는 주입니다.


In [17]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 4주차는 10주 중 가장 크게 오른 줄이다. 다음 주부터는 이 값과 견준다

recsys.record(4, "행렬분해 SVD · 요인 8개", 행렬분해_점수,
              note="규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다")

print("점수판에 4주차를 기록했습니다.\n")   # \n = 한 줄 띄우기
recsys.leaderboard(upto=4)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 4 · 행렬분해 SVD · 요인 8개 · Recall@10 = 0.3918
점수판에 4주차를 기록했습니다.



,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
